# 🔍 ColPali Interpretation & Visualization Notebook

This notebook provides advanced interpretation tools for ColPali RAG retrieval results. It helps you understand:

1. **What images ColPali retrieved** for your query
2. **Why those images were relevant** through similarity mapping
3. **Token-level attention** showing which query tokens matched which image patches
4. **Visual explanations** of the retrieval process

Perfect for debugging, improving queries, and understanding your RAG system's behavior!

## 📋 Setup & Imports

First, let's import all the necessary libraries for ColPali interpretation.

In [ ]:
import torch
import os
import base64
import io
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from typing import List, Dict, Any
import pprint

# ColPali specific imports
from colpali_engine.interpretability import (
    get_similarity_maps_from_embeddings,
    plot_all_similarity_maps,
    plot_similarity_map,
)
from colpali_engine.models import ColPali, ColPaliProcessor
from colpali_engine.utils.torch_utils import get_torch_device

# VARAG imports
from varag.rag import ColpaliRAG
from varag.utils import get_model_colpali
from sentence_transformers import SentenceTransformer
import lancedb

print("✅ All imports successful!")

## ⚙️ Initialize ColPali Model & RAG System

Set up the ColPali model and initialize our RAG system for testing.

In [ ]:
# Device setup
device = get_torch_device("auto")
print(f"Using device: {device}")

# Load ColPali model and processor
model_name = "vidore/colpali-v1.3"
colpali_model, colpali_processor = get_model_colpali(model_name)

# Initialize ColPali RAG
shared_db = lancedb.connect("~/rag_demo_db")
colpali_rag = ColpaliRAG(
    colpali_model=colpali_model,
    colpali_processor=colpali_processor,
    db=shared_db,
    table_name="colpaliDemo",
)

print("✅ ColPali RAG system initialized!")

## 🔍 Basic Query & Retrieval

Enter your query and see what ColPali retrieves.

In [ ]:
# Configure your query here
query = "What is the architecture of the model?"
top_k = 3

print(f"🔍 Query: {query}")
print(f"📊 Retrieving top {top_k} results...")

# Perform retrieval
results = colpali_rag.search(query, k=top_k)

print(f"✅ Found {len(results)} results")

# Display basic info about retrieved results
for i, result in enumerate(results, 1):
    print(f"\n📄 Result {i}:")
    print(f"   Document: {result.get('document_name', 'Unknown')}")
    print(f"   Page: {result.get('page_number', 'Unknown')}")
    if 'page_text' in result and result['page_text']:
        text_preview = result['page_text'][:100] + "..." if len(result['page_text']) > 100 else result['page_text']
        print(f"   Text preview: {text_preview}")
    else:
        print(f"   Text: No text available")

## 🖼️ Display Retrieved Images

Let's visualize the images that ColPali found relevant to your query.

In [ ]:
def base64_to_pil(base64_str: str) -> Image.Image:
    """Convert base64 string to PIL Image"""
    return Image.open(io.BytesIO(base64.b64decode(base64_str)))

# Convert base64 images to PIL Images
retrieved_images = []
for result in results:
    if 'image' in result:
        img = base64_to_pil(result['image'])
        retrieved_images.append(img)

# Display images in a grid
if retrieved_images:
    fig, axes = plt.subplots(1, len(retrieved_images), figsize=(5 * len(retrieved_images), 5))
    if len(retrieved_images) == 1:
        axes = [axes]  # Make it iterable for single image
    
    for i, (img, ax) in enumerate(zip(retrieved_images, axes)):
        ax.imshow(img)
        ax.set_title(f"Result {i+1}: {results[i].get('document_name', 'Unknown')}\nPage {results[i].get('page_number', '?')}")
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("❌ No images found in results")

## 🎯 Advanced Similarity Analysis

Now let's dive deeper and understand WHY ColPali selected these images by analyzing token-level similarity maps.

In [ ]:
def analyze_colpali_similarity(image: Image.Image, query: str, model, processor):
    """Analyze ColPali similarity maps for a given image and query"""
    
    # Preprocess inputs
    batch_images = processor.process_images([image]).to(device)
    batch_queries = processor.process_queries([query]).to(device)
    
    # Forward passes
    with torch.no_grad():
        image_embeddings = model.forward(**batch_images)
        query_embeddings = model.forward(**batch_queries)
    
    # Get the number of image patches
    n_patches = processor.get_n_patches(image_size=image.size, patch_size=model.patch_size)
    
    # Get the tensor mask to filter out the embeddings that are not related to the image
    image_mask = processor.get_image_mask(batch_images)
    
    # Generate the similarity maps
    batched_similarity_maps = get_similarity_maps_from_embeddings(
        image_embeddings=image_embeddings,
        query_embeddings=query_embeddings,
        n_patches=n_patches,
        image_mask=image_mask,
    )
    
    # Get the similarity map for our input image
    similarity_maps = batched_similarity_maps[0]  # (query_length, n_patches_x, n_patches_y)
    
    # Tokenize the query for analysis
    query_tokens = processor.tokenizer.tokenize(query)
    
    return similarity_maps, query_tokens, n_patches

print("🔧 Similarity analysis function ready!")

## 📊 Analyze First Retrieved Image

Let's analyze the top result in detail.

In [ ]:
if retrieved_images:
    # Analyze the first (top) result
    top_image = retrieved_images[0]
    
    print(f"🔍 Analyzing top result for query: '{query}'")
    print(f"📄 Document: {results[0].get('document_name', 'Unknown')}")
    print(f"📃 Page: {results[0].get('page_number', 'Unknown')}")
    
    # Perform similarity analysis
    similarity_maps, query_tokens, n_patches = analyze_colpali_similarity(
        top_image, query, colpali_model, colpali_processor
    )
    
    print(f"📐 Image patches: {n_patches}")
    print(f"🔤 Query tokens ({len(query_tokens)}): {query_tokens}")
    print(f"📊 Similarity maps shape: {similarity_maps.shape}")
    
    # Show token importance
    print("\n🎯 Token Index Reference:")
    for idx, token in enumerate(query_tokens):
        max_sim = similarity_maps[idx].max().item()
        print(f"  {idx:2d}: '{token:15s}' - Max similarity: {max_sim:.3f}")
        
else:
    print("❌ No images available for analysis")

## 🎨 Visualize Token Attention Maps

Generate attention heatmaps showing where each query token focuses on the image.

In [ ]:
if retrieved_images and 'similarity_maps' in locals():
    # Generate all similarity maps for visualization
    plots = plot_all_similarity_maps(
        image=top_image,
        query_tokens=query_tokens,
        similarity_maps=similarity_maps,
    )
    
    print(f"🎨 Generated {len(plots)} similarity map visualizations")
    
    # Display the plots
    for idx, (fig, ax) in enumerate(plots[:10]):  # Limit to first 10 tokens
        if idx < len(query_tokens):
            max_sim = similarity_maps[idx].max().item()
            ax.set_title(f"Token #{idx}: '{query_tokens[idx]}' (Max Sim: {max_sim:.3f})")
        plt.show()
        
        # Clear the figure to free memory
        plt.close(fig)
        
    print("✅ All similarity maps displayed!")
else:
    print("❌ No similarity data available for visualization")

## 🔍 Focus on Specific Tokens

Choose specific tokens to analyze in detail.

In [ ]:
if retrieved_images and 'similarity_maps' in locals():
    # Find the most important tokens (highest max similarity)
    token_importance = []
    for idx, token in enumerate(query_tokens):
        max_sim = similarity_maps[idx].max().item()
        token_importance.append((idx, token, max_sim))
    
    # Sort by importance
    token_importance.sort(key=lambda x: x[2], reverse=True)
    
    print("🏆 Top 5 Most Important Tokens:")
    for rank, (idx, token, max_sim) in enumerate(token_importance[:5], 1):
        print(f"  {rank}. Token #{idx}: '{token}' - Max similarity: {max_sim:.3f}")
    
    # Visualize the top 3 most important tokens
    print("\n🎨 Detailed visualization of top 3 tokens:")
    
    for rank, (token_idx, token, max_sim) in enumerate(token_importance[:3], 1):
        current_similarity_map = similarity_maps[token_idx]
        
        fig, ax = plot_similarity_map(
            image=top_image,
            similarity_map=current_similarity_map,
            figsize=(10, 8),
            show_colorbar=True,
        )
        
        ax.set_title(
            f"🏆 Rank {rank} - Token #{token_idx}: '{token}'\n"
            f"Max Similarity: {max_sim:.3f}",
            fontsize=14,
            fontweight='bold'
        )
        
        plt.show()
        plt.close(fig)
        
else:
    print("❌ No similarity data available")

## 📈 Comparative Analysis

Compare similarity patterns across all retrieved images.

In [ ]:
if len(retrieved_images) > 1:
    print(f"📊 Comparing similarity patterns across {len(retrieved_images)} retrieved images...")
    
    # Analyze all retrieved images
    all_analyses = []
    for i, img in enumerate(retrieved_images):
        sim_maps, tokens, patches = analyze_colpali_similarity(
            img, query, colpali_model, colpali_processor
        )
        
        # Calculate overall similarity score (max similarity across all tokens)
        overall_score = sim_maps.max().item()
        
        all_analyses.append({
            'rank': i + 1,
            'image': img,
            'similarity_maps': sim_maps,
            'overall_score': overall_score,
            'document': results[i].get('document_name', 'Unknown'),
            'page': results[i].get('page_number', 'Unknown')
        })
    
    # Display comparison summary
    print("\n🏆 Retrieval Ranking Analysis:")
    for analysis in all_analyses:
        print(f"  Rank {analysis['rank']}: {analysis['document']} (Page {analysis['page']}) - "
              f"Max Similarity: {analysis['overall_score']:.3f}")
    
    # Create a side-by-side comparison of the top 2 results
    if len(all_analyses) >= 2:
        print("\n🔍 Side-by-side comparison of top 2 results:")
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        
        for i, analysis in enumerate(all_analyses[:2]):
            # Original image
            axes[i, 0].imshow(analysis['image'])
            axes[i, 0].set_title(f"Rank {analysis['rank']}: {analysis['document']}\n"
                               f"Page {analysis['page']}")
            axes[i, 0].axis('off')
            
            # Best token's similarity map
            best_token_idx = analysis['similarity_maps'].max(dim=-1)[0].max(dim=-1)[1].item()
            best_similarity_map = analysis['similarity_maps'][best_token_idx]
            
            im = axes[i, 1].imshow(best_similarity_map.cpu().numpy(), cmap='hot', alpha=0.7)
            axes[i, 1].imshow(analysis['image'], alpha=0.3)
            axes[i, 1].set_title(f"Best Token: '{query_tokens[best_token_idx]}'\n"
                               f"Max Sim: {analysis['overall_score']:.3f}")
            axes[i, 1].axis('off')
            
            # Add colorbar
            plt.colorbar(im, ax=axes[i, 1], fraction=0.046, pad=0.04)
        
        plt.tight_layout()
        plt.show()
        plt.close(fig)
        
else:
    print("ℹ️ Only one image retrieved - no comparison available")

## 💡 Query Optimization Suggestions

Based on the analysis, generate suggestions for improving your query.

In [ ]:
if 'token_importance' in locals():
    print("💡 Query Optimization Suggestions:")
    print("=" * 50)
    
    # Analyze token effectiveness
    high_impact_tokens = [t for t in token_importance if t[2] > 0.5]
    low_impact_tokens = [t for t in token_importance if t[2] < 0.1]
    
    if high_impact_tokens:
        print("\n✅ HIGH IMPACT TOKENS (working well):")
        for idx, token, sim in high_impact_tokens:
            print(f"   • '{token}' (similarity: {sim:.3f})")
        print("   → These tokens are effective, consider using similar terms")
    
    if low_impact_tokens:
        print("\n⚠️ LOW IMPACT TOKENS (might need refinement):")
        for idx, token, sim in low_impact_tokens:
            print(f"   • '{token}' (similarity: {sim:.3f})")
        print("   → Consider replacing these with more specific terms")
    
    # General suggestions
    print("\n🎯 GENERAL SUGGESTIONS:")
    print("   • Focus on visual elements: diagrams, charts, figures")
    print("   • Use technical terms that appear in document layouts")
    print("   • Consider document structure: headers, captions, tables")
    print("   • Be specific about what you're looking for visually")
    
    # Query variants
    print("\n🔄 QUERY VARIANTS TO TRY:")
    if any("model" in token.lower() for _, token, _ in token_importance):
        print("   • 'architecture diagram' or 'model structure'")
    if any("figure" in token.lower() for _, token, _ in token_importance):
        print("   • 'Figure X' or 'diagram showing'")
    print("   • Try more specific technical terms")
    print("   • Include visual cues like 'chart', 'table', 'image'")
    
else:
    print("ℹ️ No token analysis available for suggestions")

print("\n" + "=" * 50)
print("🎉 ColPali Interpretation Analysis Complete!")
print("Feel free to modify the query above and re-run the analysis.")

## 🔧 Interactive Query Testing

Use this cell to quickly test different queries and see their impact.

In [ ]:
# Test different queries quickly
test_queries = [
    "model architecture",
    "training process diagram",
    "performance results table",
    "experimental setup figure",
    "comparison chart"
]

print("🧪 Quick Query Testing:")
print("=" * 40)

for test_query in test_queries:
    print(f"\n🔍 Testing: '{test_query}'")
    
    # Quick retrieval test
    test_results = colpali_rag.search(test_query, k=1)
    
    if test_results:
        result = test_results[0]
        print(f"   📄 Top result: {result.get('document_name', 'Unknown')} (Page {result.get('page_number', '?')})")
        
        # Quick similarity analysis
        if 'image' in result:
            img = base64_to_pil(result['image'])
            sim_maps, tokens, _ = analyze_colpali_similarity(
                img, test_query, colpali_model, colpali_processor
            )
            max_sim = sim_maps.max().item()
            print(f"   📊 Max similarity: {max_sim:.3f}")
            
            # Find best token
            best_token_idx = sim_maps.max(dim=-1)[0].max(dim=-1)[1].item()
            if best_token_idx < len(tokens):
                print(f"   🎯 Best token: '{tokens[best_token_idx]}'")
    else:
        print("   ❌ No results found")

print("\n" + "=" * 40)
print("✅ Quick testing complete!")

## 📝 Summary & Next Steps

### What We Learned:

1. **Retrieval Quality**: How well ColPali matched your query
2. **Token Importance**: Which words in your query were most effective
3. **Visual Attention**: Where the model focused on each image
4. **Ranking Logic**: Why certain results ranked higher than others

### Best Practices for ColPali Queries:

- ✅ Use **visual descriptors** (diagram, chart, figure, table)
- ✅ Be **specific** about what you're looking for
- ✅ Include **technical terms** that might appear in documents
- ✅ Consider **document structure** (headers, captions, labels)
- ❌ Avoid overly generic terms
- ❌ Don't rely solely on text that might not be visible

### Next Steps:

1. **Refine your queries** based on the token analysis
2. **Test with different document types** to understand patterns
3. **Experiment with query length** and complexity
4. **Use similarity maps** to debug unexpected results

---

*Happy querying with ColPali! 🎉*